# Predicting XRP Price using Random Forest Regression
**Course:** CCE3131 - AI/ML Lab (Mini Project) &nbsp;|&nbsp; **Student:** Ritvik Chauhan &nbsp;|&nbsp; **Department of Computer and Communication Engineering, Manipal University Jaipur**

**What this does:** it predicts the next-day closing price of XRP (from the Kaggle file `XRPUSDT20182024new.csv`) with a **Random Forest** regressor.
The forest is tuned with time-ordered cross-validation (n_estimators, max_depth, min_samples_leaf), compared with the default forest and with a naive "tomorrow equals today" baseline,
and tested one day at a time on days it has never seen.

**How to run:** `Runtime -> Run all`. The first cell asks you to upload `XRPUSDT20182024new.csv`. The last cell writes the project report (.docx and .pdf) and downloads both.

In [ ]:
import os
try:
    from google.colab import files
    if not os.path.exists('XRPUSDT20182024new.csv'):
        print('Select XRPUSDT20182024new.csv from your computer...')
        files.upload()
except ImportError:
    pass
print('CSV found:', os.path.exists('XRPUSDT20182024new.csv'))

In [ ]:
!pip install -q python-docx pypdf
import warnings, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon, Ellipse
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.base import clone

plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
BLUE, ORANGE, GREEN, GREY = "#1f4e79", "#c55a11", "#548235", "#7f7f7f"
os.makedirs("figures", exist_ok=True)
np.random.seed(42)
def save(fig, name):
    fig.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight"); plt.show()

## 1. Configuration

In [ ]:
DATA_PATH = "XRPUSDT20182024new.csv"       # daily XRP/USDT prices downloaded from Kaggle
DATASET_TITLE = ("XRPUSDT20182024new: daily XRP/USDT prices from 26 May 2018 to 26 May 2024 (Date, Price, Open, High, Low, Vol, Change %). "
                 "Kaggle dataset by kryusufkaya")
DATASET_URL   = "https://www.kaggle.com/datasets/kryusufkaya/xrpusdt20182024new"

ASSET     = "XRP"           # name used in the report
TICKER    = "XRP/USDT"      # label used in the report
TEST_DAYS = 90              # the last 90 days are used for testing
PRICE_DEC = 4               # decimals shown for prices (XRP is below 2 USD)

## 2. Load and check the data

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("Shape:", raw.shape)
raw.head(3)

In [ ]:
miss = raw.isna().sum()
print("Missing values:", int(miss.sum()))
print("Duplicate rows:", int(raw.duplicated().sum()), "| duplicate dates:", int(raw["Date"].duplicated().sum()))
print(raw.dtypes)

In [ ]:
# Clean: proper dates, oldest first, rename the columns, make sure every price column is a number
df = raw.rename(columns={"Price": "Close", "Vol": "Volume"}).drop(columns=["Change %"], errors="ignore")
df["Date"] = pd.to_datetime(df["Date"])
df = df.drop_duplicates("Date").sort_values("Date").set_index("Date")
for c in ["Open", "High", "Low", "Close", "Volume"]:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", ""), errors="coerce")
dup_days = int(raw["Date"].duplicated().sum())

df = df.asfreq("D")                               # one row for every calendar day
filled_days = int(df.isna().any(axis=1).sum())    # days that were missing in the file
df = df.ffill()                                   # fill them with the previous day
DATA_SOURCE_TEXT = ('taken from the Kaggle dataset "XRPUSDT20182024new", a CSV file with the columns Date, Price, Open, High, Low, Vol and Change %. '
                    'The Price column is used as the closing price.')
print(df.shape, df.index.min().date(), "->", df.index.max().date(), "| missing days filled:", filled_days)
df.tail()

## 3. Returns and features
Prices drift up and down for years, so instead of the price itself I predict the **daily return** (percentage change), which stays in a steady range. The predicted price is then `today's price x (1 + predicted return)`.

In [ ]:
price = df["Close"]
ret = np.log(price).diff().dropna()               # daily log-return
n_obs = len(price); test_n = TEST_DAYS; train_n = n_obs - test_n

# Features: only what is known at the END of day t (no leakage)
feat = pd.DataFrame({f"lag{i}": ret.shift(i - 1) for i in range(1, 6)})   # return of today and the 4 days before
feat["rm7"]  = ret.rolling(7).mean();  feat["rm14"] = ret.rolling(14).mean()   # average return, last 7 / 14 days
feat["rs7"]  = ret.rolling(7).std();   feat["rs14"] = ret.rolling(14).std()    # volatility, last 7 / 14 days
FEATURES = list(feat.columns)
data = pd.concat([feat, (ret.shift(-1) * 100).rename("target")], axis=1).dropna()   # target = next-day return in %
print(len(FEATURES), "features |", len(data), "rows")
data.head(3)

## 4. Exploratory analysis

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
ax[0].plot(price.index[:train_n], price.iloc[:train_n], label="Train", color=BLUE)
ax[0].plot(price.index[train_n:], price.iloc[train_n:], label="Test", color=ORANGE)
ax[0].axvline(price.index[train_n], color=GREY, ls="--")
ax[0].set_title(f"{ASSET} ({TICKER}) daily closing price"); ax[0].set_ylabel("USD"); ax[0].legend()
ax[1].plot(ret.rolling(30).std() * 100, color=ORANGE); ax[1].set_title("30-day rolling volatility of daily returns"); ax[1].set_ylabel("%")
fig.tight_layout(); save(fig, "fig_3_1")

In [ ]:
i_tr = data.index.get_loc(price.index[train_n - 1])            # last training row
train_rows = data.iloc[:i_tr]
corr = train_rows.corr()["target"].drop("target")
corr_table = corr.round(3); display(corr_table.to_frame("corr with next-day return"))
print("Largest absolute correlation:", corr.abs().idxmax(), round(corr.abs().max(), 3))

fig, ax = plt.subplots(1, 3, figsize=(14, 3.9))
r_pct = ret * 100
ax[0].hist(r_pct, bins=60, density=True, color=BLUE, alpha=.8)
xs = np.linspace(r_pct.min(), r_pct.max(), 200)
ax[0].plot(xs, np.exp(-.5 * ((xs - r_pct.mean()) / r_pct.std()) ** 2) / (r_pct.std() * np.sqrt(2 * np.pi)), color=ORANGE, label="Normal curve"); ax[0].legend()
ax[0].set_title("Distribution of daily returns (%)")
ax[1].scatter(train_rows["lag1"] * 100, train_rows["target"], s=5, alpha=.4, color=BLUE)
m_, b_ = np.polyfit(train_rows["lag1"] * 100, train_rows["target"], 1)
xx = np.linspace(train_rows["lag1"].min() * 100, train_rows["lag1"].max() * 100, 50); ax[1].plot(xx, m_ * xx + b_, color=ORANGE)
ax[1].set_xlabel("Today's return (%)"); ax[1].set_ylabel("Next-day return (%)"); ax[1].set_title("Today vs next-day return")
ax[2].barh(corr.index[::-1], corr.values[::-1], color=[GREEN if c > 0 else ORANGE for c in corr.values[::-1]])
ax[2].axvline(0, color="black", lw=.8); ax[2].set_title("Correlation of each feature with next-day return")
fig.tight_layout(); save(fig, "fig_3_2")

## 5. Train / test split and evaluation helpers

In [ ]:
y = price.values.astype(float)
actual = y[train_n:]; prev = y[train_n - 1:-1]
print(f"Train: {train_n:,} days  ({price.index[0].date()} to {price.index[train_n-1].date()})")
print(f"Test : {test_n} days   ({price.index[train_n].date()} to {price.index[-1].date()})")

def evaluate(pred):
    mae = mean_absolute_error(actual, pred); mse = mean_squared_error(actual, pred)
    mape = float(np.mean(np.abs((actual - pred) / actual)) * 100)
    pc = np.sign(pred - prev)
    dir_acc = float(np.mean(pc == np.sign(actual - prev)) * 100) if np.any(pc != 0) else float("nan")
    return dict(MAE=float(mae), MSE=float(mse), RMSE=float(np.sqrt(mse)), MAPE=mape, DirAcc=dir_acc)

preds, metrics = {}, {}
preds["Naive (last value)"] = prev.copy()           # baseline: tomorrow's price = today's price
metrics["Naive (last value)"] = evaluate(preds["Naive (last value)"]); print("Naive baseline", {a: round(b, 5) for a, b in metrics["Naive (last value)"].items()})

## 6. Random Forest and hyperparameter tuning
A **Random Forest** is a group of decision trees. Each tree is trained on a random sample of the days and looks at a random subset of the features at every split; the forest's prediction is the **average** of all the trees. Averaging many different trees reduces the over-fitting of a single deep tree.

Three hyperparameters matter most: `n_estimators` (number of trees), `max_depth` (how deep each tree may grow) and `min_samples_leaf` (minimum number of days in a leaf). I score every combination with **5-fold cross-validation that respects time order** (`TimeSeriesSplit`): each fold trains on an earlier part of the training period and is scored on the part that follows, so the forest never trains on the future. The scores are MAE and MSE of the next-day return in %.

In [ ]:
X_tr, y_tr = train_rows[FEATURES], train_rows["target"]
tscv = TimeSeriesSplit(n_splits=5)
grid = {"n_estimators": [100, 200], "max_depth": [2, 3, 5, 8, 12], "min_samples_leaf": [5, 20, 50]}

gs = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), grid, cv=tscv,
                  scoring={"mse": "neg_mean_squared_error", "mae": "neg_mean_absolute_error"}, refit="mse").fit(X_tr, y_tr)
cvres = pd.DataFrame(gs.cv_results_)
cvres["cv_mse"] = -cvres["mean_test_mse"]; cvres["cv_mae"] = -cvres["mean_test_mae"]
for c in ["n_estimators", "max_depth", "min_samples_leaf"]: cvres[c] = cvres["param_" + c].astype(int)
best_params = {k: int(v) for k, v in gs.best_params_.items()}
print("Best settings:", best_params)

# reference points: default forest and "always predict a 0% return"
def cv_scores(make):
    m = [(mean_squared_error(y_tr.iloc[v], make(t, v)), mean_absolute_error(y_tr.iloc[v], make(t, v))) for t, v in tscv.split(X_tr)]
    return float(np.mean([a for a, _ in m])), float(np.mean([b for _, b in m]))
default_cv = cv_scores(lambda t, v: RandomForestRegressor(random_state=42, n_jobs=-1).fit(X_tr.iloc[t], y_tr.iloc[t]).predict(X_tr.iloc[v]))
base_cv = (float(np.mean([np.mean(y_tr.iloc[v] ** 2) for t, v in tscv.split(X_tr)])), float(np.mean([np.mean(np.abs(y_tr.iloc[v])) for t, v in tscv.split(X_tr)])))
top8 = cvres.sort_values("cv_mse").head(8)[["n_estimators", "max_depth", "min_samples_leaf", "cv_mse", "cv_mae"]].reset_index(drop=True)
display(top8.round(4))
print(f"Default forest (100 trees, no depth limit, min_samples_leaf=1): CV MSE = {default_cv[0]:.3f}, CV MAE = {default_cv[1]:.3f}")
print(f"Always predict 0% return                                   : CV MSE = {base_cv[0]:.3f}, CV MAE = {base_cv[1]:.3f}")

In [ ]:
# Effect of each hyperparameter on the cross-validation error
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for a, col in zip(ax, ["max_depth", "min_samples_leaf", "n_estimators"]):
    g = cvres.groupby(col)["cv_mse"].mean()
    a.plot(g.index, g.values, marker="o", color=BLUE); a.axhline(base_cv[0], color=ORANGE, ls="--", label="Always predict 0%")
    a.set_xlabel(col); a.set_ylabel("CV MSE (%$^2$)"); a.set_title(f"Effect of {col}")
    if col == "n_estimators": a.set_xticks(g.index)
ax[0].legend(fontsize=8)
fig.tight_layout(); save(fig, "fig_3_3")

### Walk-forward testing (one day ahead)
For every test day the forest is trained only on earlier days, predicts the next-day return, and then the real value is added to the history. To save time the forest is refitted every 5 days. The **default** forest and the **tuned** forest are tested in exactly the same way.

In [ ]:
def run_rf(params, refit_every=5):
    p, mdl = [], None
    for k, t in enumerate(range(train_n - 1, n_obs - 1)):
        i = data.index.get_loc(price.index[t])
        if mdl is None or k % refit_every == 0:
            mdl = RandomForestRegressor(random_state=42, n_jobs=-1, **params).fit(data.iloc[:i][FEATURES], data.iloc[:i]["target"])
        p.append(y[t] * np.exp(mdl.predict(data.iloc[[i]][FEATURES])[0] / 100))
    return np.array(p)

DEF, TUN = "Random Forest (default)", "Random Forest (tuned)"
preds[DEF] = run_rf({});            metrics[DEF] = evaluate(preds[DEF])
preds[TUN] = run_rf(best_params);   metrics[TUN] = evaluate(preds[TUN])
for k in (DEF, TUN): print(k, {a: round(b, 5) for a, b in metrics[k].items()})

## 7. Results

In [ ]:
table = pd.DataFrame(metrics).T.sort_values("RMSE")
table.columns = ["MAE (USD)", "MSE (USD^2)", "RMSE (USD)", "MAPE (%)", "Directional acc. (%)"]
display(table.round({"MAE (USD)": 4, "MSE (USD^2)": 6, "RMSE (USD)": 4, "MAPE (%)": 2, "Directional acc. (%)": 1}))
best_model = table.index[0]; top = TUN
print("Lowest RMSE overall:", best_model)

dates = price.index[train_n:]
fig, ax = plt.subplots()
ax.plot(dates, actual, color="black", lw=2, label="Actual")
ax.plot(dates, preds["Naive (last value)"], lw=1.2, color=GREY, label="Naive (last value)")
ax.plot(dates, preds[DEF], lw=1.2, color=GREEN, label=DEF); ax.plot(dates, preds[TUN], lw=1.4, color=ORANGE, label=TUN)
ax.set_title("Actual vs predicted closing price (test period)"); ax.set_ylabel("USD"); ax.legend(ncol=2)
save(fig, "fig_3_4")

order_ = table.index.tolist()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for a, col in [(ax[0], "RMSE (USD)"), (ax[1], "MAE (USD)")]:
    a.barh(order_[::-1], table[col][::-1], color=BLUE); a.set_title(col + " - lower is better")
    for i_, v in enumerate(table[col][::-1]): a.text(v, i_, f" {v:.4f}", va="center", fontsize=9)
fig.tight_layout(); save(fig, "fig_3_5")

### Wins against the naive forecast, direction of the move and error distribution

In [ ]:
err = {k: actual - preds[k] for k in preds}
win_days = int(np.sum(np.abs(err[top]) < np.abs(err["Naive (last value)"])))
print(f"{top} was closer than the naive forecast on {win_days} of {test_n} days")

pdir = np.sign(preds[top] - prev) > 0; adir = np.sign(actual - prev) > 0
dir_conf = dict(uu=int(np.sum(adir & pdir)), ud=int(np.sum(adir & ~pdir)), du=int(np.sum(~adir & pdir)), dd=int(np.sum(~adir & ~pdir)))
dir_margin = float(1.96 * np.sqrt(.25 / test_n) * 100)       # luck band of a coin toss
print(dir_conf, f"| a coin toss gives 50% +/- {dir_margin:.0f} points over {test_n} days")

def err_stats(k):
    e = err[k]; rel = np.abs(e) / actual * 100
    return dict(bias=float(e.mean()), std=float(e.std()), max=float(np.abs(e).max()),
                w1=float((rel <= 1).mean() * 100), w2=float((rel <= 2).mean() * 100), w5=float((rel <= 5).mean() * 100))
err_table = {k: err_stats(k) for k in [top, "Naive (last value)"]}
display(pd.DataFrame(err_table).round(4))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].hist(err[top], bins=25, color=BLUE, alpha=.85); ax[0].axvline(0, color=ORANGE); ax[0].set_title(f"Forecast error (USD), {top}")
ax[1].plot(dates, err[top], color=BLUE, lw=1); ax[1].axhline(0, color=ORANGE); ax[1].set_title("Forecast error over the test period (USD)")
fig.tight_layout(); save(fig, "fig_3_6")

### Which features does the forest rely on? (feature importance)

In [ ]:
rf_fit = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params).fit(X_tr, y_tr)
imp = pd.Series(rf_fit.feature_importances_ * 100, index=FEATURES).sort_values(ascending=False)
coefs = [dict(feature=f, imp=float(v), corr=float(corr[f])) for f, v in imp.items()]
display(pd.DataFrame(coefs).round(3))
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(imp.index[::-1], imp.values[::-1], color=BLUE); ax.set_xlabel("Importance (%)"); ax.set_title("Random Forest feature importance (tuned forest)")
save(fig, "fig_3_7")

## 8. Next-day forecast
The tuned forest is refitted on all the data and used to predict the day after the last date in the file. The interval is taken from the 5th and 95th percentile of the percentage errors on the test days.

In [ ]:
final = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params).fit(data[FEATURES], data["target"])
last_feat = feat.dropna().iloc[[-1]]
p_next = float(y[-1] * np.exp(final.predict(last_feat[FEATURES])[0] / 100))
rel_err = actual / preds[top] - 1
lo_, hi_ = p_next * (1 + np.percentile(rel_err, 5)), p_next * (1 + np.percentile(rel_err, 95))
next_date = price.index[-1] + pd.Timedelta(days=1)
print(f"{top}: forecast for {next_date.date()} = {p_next:.{PRICE_DEC}f} USD (90% interval {lo_:.{PRICE_DEC}f} to {hi_:.{PRICE_DEC}f}); last close = {y[-1]:.{PRICE_DEC}f}")

fig, ax = plt.subplots()
ax.plot(price.index[-60:], price.iloc[-60:], color=BLUE, label="Last 60 days")
ax.errorbar([next_date], [p_next], yerr=[[p_next - lo_], [hi_ - p_next]], fmt="o", color=ORANGE, capsize=6, label="Next-day forecast (90% interval)")
ax.set_title(f"Next-day forecast of {ASSET} (Random Forest)"); ax.set_ylabel("USD"); ax.legend()
save(fig, "fig_3_8")

### Try it yourself (interactive)
Pick any day from the test period. The table shows what the forests and the naive baseline predicted for that day, using only the data before it, next to the real price.

In [ ]:
try:
    import ipywidgets as w
    from IPython.display import display, clear_output
    day = w.Dropdown(options=[str(d.date()) for d in dates], value=str(dates[len(dates) // 2].date()), description="Test day",
                     style={"description_width": "80px"}, layout=w.Layout(width="260px"))
    out = w.Output()
    def show(_=None):
        with out:
            clear_output(wait=True)
            j = [str(d.date()) for d in dates].index(day.value)
            t = pd.DataFrame({"Predicted price": {k: preds[k][j] for k in preds}})
            t["Error"] = t["Predicted price"] - actual[j]
            print(f"Actual closing price on {day.value}: {actual[j]:.{PRICE_DEC}f} USD   (previous day: {prev[j]:.{PRICE_DEC}f})")
            display(t.round(PRICE_DEC))
    day.observe(show, "value"); display(w.VBox([day, out])); show()
except ImportError:
    print("ipywidgets is not installed here; skip this cell.")

## 9. Diagrams for the report (architecture and flow chart)

In [ ]:
def box(ax, x, y, w, h, title, lines, fc="#dce9f7"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08", fc=fc, ec=BLUE, lw=1.4))
    ax.text(x + w / 2, y + h - .22, title, ha="center", va="top", fontsize=9.5, fontweight="bold", color=BLUE)
    ax.text(x + w / 2, y + h - .62, "\n".join(lines), ha="center", va="top", fontsize=7.8)

fig, ax = plt.subplots(figsize=(13, 3.2)); ax.axis("off"); ax.set_xlim(0, 13); ax.set_ylim(0, 3)
stages = [("Input:\nXRP price CSV", ["Kaggle file", "pandas, NumPy", f"{TICKER} daily prices"]),
          ("Data\nPre-processing", ["Daily calendar", "Fill missing days", "Daily returns"]),
          ("Feature\nEngineering", ["5 lagged returns", "7/14-day average", "7/14-day volatility"]),
          ("Random Forest\nTraining and Tuning", ["Default forest", "Grid search", "Time-series CV"]),
          ("Evaluation and\nForecast", ["Walk-forward test", "MAE, MSE, MAPE", "Next-day forecast"])]
for i, (t, l) in enumerate(stages):
    x = .15 + i * 2.55; box(ax, x, .2, 2.15, 2.6, t, l, "#dce9f7" if i else "#fce4d6")
    if i < 4: ax.annotate("", xy=(x + 2.5, 1.5), xytext=(x + 2.17, 1.5), arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=1.6))
save(fig, "arch")

TEAL, AMBER, LTEAL, LAMB = "#0f6b6b", "#b7791f", "#d9efee", "#fdf0d5"
fig, ax = plt.subplots(figsize=(6.6, 10.4)); ax.axis("off"); ax.set_xlim(0, 10.6); ax.set_ylim(0, 20.6)
CX, W = 4.0, 5.2
def node(kind, y, text, w=W, h=.82, x=CX, fc=LTEAL, ec=TEAL):
    if kind == "oval":
        ax.add_patch(Ellipse((x, y), w * .62, h, fc=LAMB, ec=AMBER, lw=1.6))
    elif kind == "io":
        k = .35; ax.add_patch(Polygon([(x - w/2 + k, y + h/2), (x + w/2 + k, y + h/2), (x + w/2 - k, y - h/2), (x - w/2 - k, y - h/2)], fc=LAMB, ec=AMBER, lw=1.6))
    elif kind == "dia":
        ax.add_patch(Polygon([(x, y + h), (x + w/2, y), (x, y - h), (x - w/2, y)], fc="#fff", ec=AMBER, lw=1.8))
    else:
        ax.add_patch(FancyBboxPatch((x - w/2, y - h/2), w, h, boxstyle="round,pad=0.02,rounding_size=0.12", fc=fc, ec=ec, lw=1.4))
    ax.text(x, y, text, ha="center", va="center", fontsize=8.2, fontweight="bold" if kind == "oval" else "normal")
def arrow(x1, y1, x2, y2, label=None, lx=0, ly=0):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color="#444", lw=1.2))
    if label: ax.text((x1 + x2) / 2 + lx, (y1 + y2) / 2 + ly, label, fontsize=8, color=AMBER, fontweight="bold")
def line(xs, ys): ax.plot(xs, ys, color="#444", lw=1.2)

Y = dict(start=20, load=18.7, check=16.6, ret=14.3, feat=13.0, split=11.7, tune=10.4, pred=9.1, add=7.8, more=6.0, metr=4.0, fore=2.7, rep=1.5)
node("oval", Y["start"], "START", h=.8)
node("io", Y["load"], f"Load daily {TICKER} prices (CSV)")
node("dia", Y["check"], "Any missing\ndays in the file?", w=4.6, h=1.0)
node("rect", Y["check"], "Fill from the\nprevious day", w=2.7, h=.9, x=8.6)
node("rect", Y["ret"], "Calculate daily returns")
node("rect", Y["feat"], "Build 9 features (lags, averages, volatility)")
node("rect", Y["split"], f"Split by time: train | last {test_n} days test")
node("rect", Y["tune"], "Tune the Random Forest (time-series CV)")
node("rect", Y["pred"], "Predict day t+1 with the forest", fc=LAMB, ec=AMBER)
node("rect", Y["add"], "Add the real price to history, refit forest", fc=LAMB, ec=AMBER)
node("dia", Y["more"], "More test days left?", w=4.6, h=1.0)
node("rect", Y["metr"], "MAE, MSE, RMSE, MAPE, direction accuracy")
node("rect", Y["fore"], "Feature importance; next-day forecast")
node("io", Y["rep"], "Write the report")
node("oval", .45, "END", h=.7)

arrow(CX, Y["start"] - .4, CX, Y["load"] + .41); arrow(CX, Y["load"] - .41, CX, Y["check"] + 1.0)
arrow(CX, Y["check"] - 1.0, CX, Y["ret"] + .41, "No", .15, .05)
arrow(CX + 2.3, Y["check"], 7.25, Y["check"], "Yes", -.1, .15)
line([8.6, 8.6], [Y["check"] - .45, Y["ret"]]); arrow(8.6, Y["ret"], CX + W / 2, Y["ret"])
for a_, b_ in [("ret", "feat"), ("feat", "split"), ("split", "tune"), ("tune", "pred"), ("pred", "add")]:
    arrow(CX, Y[a_] - .41, CX, Y[b_] + .41)
arrow(CX, Y["add"] - .41, CX, Y["more"] + 1.0)
arrow(CX, Y["more"] - 1.0, CX, Y["metr"] + .41, "No", .15, .05)
line([CX + 2.3, 9.9], [Y["more"], Y["more"]]); line([9.9, 9.9], [Y["more"], Y["pred"]])
arrow(9.9, Y["pred"], CX + W / 2, Y["pred"]); ax.text(7.2, Y["more"] + .15, "Yes: next day", fontsize=8, color=AMBER, fontweight="bold")
arrow(CX, Y["metr"] - .41, CX, Y["fore"] + .41); arrow(CX, Y["fore"] - .41, CX, Y["rep"] + .41); arrow(CX, Y["rep"] - .41, CX, .8)
save(fig, "flow")

## 10. Save results

In [ ]:
yrs = []
for yr, g in price.groupby(price.index.year):
    r_ = np.log(g).diff().dropna()
    yrs.append(dict(year=int(yr), days=int(len(g)), first=float(g.iloc[0]), last=float(g.iloc[-1]), chg=float((g.iloc[-1] / g.iloc[0] - 1) * 100), vol=float(r_.std())))
def seg(name, s):
    r_ = np.log(s).diff().dropna()
    return dict(name=name, period=f"{s.index[0].date()} to {s.index[-1].date()}", days=int(len(s)), mean_close=float(s.mean()), mean_ret=float(r_.mean()), vol=float(r_.std()))
split_table = [seg("Training", price.iloc[:train_n]), seg("Test", price.iloc[train_n:]), seg("Total", price)]
sample = data.loc[price.index[train_n - 1]:].iloc[:5]
sample_rows = [dict(date=str(d.date()), close=float(price[d]), lag1=float(r.lag1), lag2=float(r.lag2), rm7=float(r.rm7), rs7=float(r.rs7), next_ret=float(r.target / 100)) for d, r in sample.iterrows()]

results = dict(
    ticker=TICKER, asset_name=ASSET, data_source_text=DATA_SOURCE_TEXT, dataset_title=DATASET_TITLE, dataset_url=DATASET_URL, price_dec=PRICE_DEC,
    dup_days=dup_days, filled_days=filled_days,
    start=str(price.index[0].date()), end=str(price.index[-1].date()), train_end=str(price.index[train_n - 1].date()), test_start=str(price.index[train_n].date()),
    n_obs=int(n_obs), train_n=int(train_n), test_n=int(test_n), last_price=float(y[-1]),
    metrics=metrics, best_model=best_model, top_model=top, default_model=DEF,
    stats=dict(min=float(price.min()), max=float(price.max()), mean=float(price.mean()), std=float(price.std()),
               ret_mean=float(ret.mean()), ret_std=float(ret.std()), max_gain=float(np.expm1(ret.max()) * 100), max_fall=float(np.expm1(ret.min()) * 100)),
    best_params=best_params, grid={k: v for k, v in grid.items()}, tuning_top=top8.to_dict("records"),
    default_cv=dict(mse=default_cv[0], mae=default_cv[1]), base_cv=dict(mse=base_cv[0], mae=base_cv[1]),
    win_days=win_days, dir_model=top, dir_conf=dir_conf, dir_margin=dir_margin,
    err_table=err_table, coefs=coefs, corr=[dict(feature=k, corr=float(v)) for k, v in corr.items()],
    next_day=dict(date=str(next_date.date()), pred=p_next, lo=float(lo_), hi=float(hi_), last=float(y[-1])),
    year_table=yrs, split_table=split_table, sample_rows=sample_rows,
)
json.dump(results, open("results.json", "w"), indent=2)
print("Saved results.json")

## 11. Generate the project report (.docx and .pdf)
The report follows the department's mini-project format. **Type your registration number in `REG_NO` below** (and change `GUIDE` if yours is different), then run the cell.
It installs LibreOffice first (about a minute) so the page numbers can be filled in, and then downloads the .docx and the .pdf.

In [ ]:
!pip install -q pypdf
!apt-get -qq install -y libreoffice-writer fonts-liberation > /dev/null 2>&1

# ---- report generator ----
import json, os, re, shutil, subprocess, base64, datetime
from docx import Document
from docx.shared import Pt, Inches, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_TAB_ALIGNMENT, WD_TAB_LEADER
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.section import WD_SECTION
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

# ------------------------------------------------------------------ student details
STUDENT_NAME = "Ritvik Chauhan"
REG_NO = "______________"                 # <- registration number
GUIDE = "Dr Sandeep Kumar Sharma"          # <- change if your guide is different
SEMESTER = "Jul – Dec 2026"
PROJECT_TITLE = "Predicting XRP Price using Random Forest Regression"
LOGO_B64 = ""                             # filled by the notebook (university logo)

FONT = "Times New Roman"


# ------------------------------------------------------------------ low-level helpers
def _font(run, size=12, bold=False, italic=False):
    run.font.name = FONT
    rPr = run._element.get_or_add_rPr()
    rPr.rFonts.set(qn("w:eastAsia"), FONT)
    run.font.size = Pt(size)
    run.bold = bold
    run.italic = italic


def _para(doc, text="", size=12, bold=False, italic=False, align="justify", after=6, before=0, line=1.25, keep=False):
    p = doc.add_paragraph()
    if text:
        r = p.add_run(text)
        _font(r, size, bold, italic)
    pf = p.paragraph_format
    pf.space_after, pf.space_before, pf.line_spacing = Pt(after), Pt(before), line
    pf.keep_with_next = keep
    p.alignment = {"center": WD_ALIGN_PARAGRAPH.CENTER, "justify": WD_ALIGN_PARAGRAPH.JUSTIFY,
                   "left": WD_ALIGN_PARAGRAPH.LEFT}[align]
    return p


def _rich(doc, parts, style=None, after=4, left=None):
    """parts = [(text, bold), ...] in one paragraph."""
    p = doc.add_paragraph(style=style) if style else doc.add_paragraph()
    for t, b in parts:
        _font(p.add_run(t), 12, bold=b)
    p.paragraph_format.space_after = Pt(after)
    p.paragraph_format.line_spacing = 1.2
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    if left is not None:
        p.paragraph_format.left_indent = Cm(left)
    return p


def _chapter(doc, text):
    p = _para(doc, text, 15, True, align="center", after=10, keep=True)
    p.paragraph_format.page_break_before = True
    return p


def _h2(doc, text):
    return _para(doc, text, 13.5, True, align="left", before=8, after=4, keep=True)


def _bullets(doc, items):
    for it in items:
        p = doc.add_paragraph(style="List Bullet")
        _font(p.add_run(it), 12)
        p.paragraph_format.space_after = Pt(2)
        p.paragraph_format.line_spacing = 1.2
        p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY


def _numbered(doc, items):
    for i, (head, body) in enumerate(items, 1):
        _rich(doc, [(f"{i}.  ", True), (head + ": ", True), (body, False)], after=4, left=0.6)


def _shade(cell, color):
    tcPr = cell._element.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear"); shd.set(qn("w:color"), "auto"); shd.set(qn("w:fill"), color)
    tcPr.append(shd)


def _no_split(row):
    trPr = row._tr.get_or_add_trPr()
    el = OxmlElement("w:cantSplit"); trPr.append(el)


def _borders(t):
    tblPr = t._tbl.tblPr
    for old in tblPr.findall(qn("w:tblBorders")):
        tblPr.remove(old)
    bd = OxmlElement("w:tblBorders")
    for side, sz, col in (("top", 12, "1F4E79"), ("bottom", 12, "1F4E79"), ("insideH", 4, "D0D7E1"),
                          ("left", 0, "FFFFFF"), ("right", 0, "FFFFFF"), ("insideV", 0, "FFFFFF")):
        el = OxmlElement("w:" + side)
        el.set(qn("w:val"), "single" if sz else "nil"); el.set(qn("w:sz"), str(sz)); el.set(qn("w:space"), "0"); el.set(qn("w:color"), col)
        bd.append(el)
    tblPr.append(bd)


def _table(doc, header, rows, widths, size=10.5, bold_rows=(), left_cols=(0,)):
    t = doc.add_table(rows=1, cols=len(header))
    t.alignment = WD_TABLE_ALIGNMENT.CENTER; t.autofit = False
    _borders(t)
    for i, h in enumerate(header):
        c = t.rows[0].cells[i]; c.text = ""
        r = c.paragraphs[0].add_run(h); _font(r, size, bold=True); r.font.color.rgb = RGBColor(255, 255, 255)
        c.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.LEFT if i in left_cols else WD_ALIGN_PARAGRAPH.CENTER
        _shade(c, "1F4E79")
    trPr = t.rows[0]._tr.get_or_add_trPr(); trPr.append(OxmlElement("w:tblHeader"))
    for ri, row in enumerate(rows):
        r = t.add_row(); _no_split(r)
        for i, v in enumerate(row):
            c = r.cells[i]; c.text = ""
            _font(c.paragraphs[0].add_run(str(v)), size, bold=(ri in bold_rows))
            c.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.LEFT if i in left_cols else WD_ALIGN_PARAGRAPH.CENTER
            if ri in bold_rows:
                _shade(c, "DDEBF7")
            elif ri % 2 == 1:
                _shade(c, "F2F6FA")
    for i, w in enumerate(widths):
        t.columns[i].width = Inches(w)
    for row in t.rows:
        for i, w in enumerate(widths):
            row.cells[i].width = Inches(w)
        for c in row.cells:
            for p in c.paragraphs:
                p.paragraph_format.space_after = Pt(2); p.paragraph_format.space_before = Pt(2)
                p.paragraph_format.keep_with_next = True
    return t


def _tcap(doc, n, title=None):
    title = title or dict(TABS)[n]
    p = doc.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.space_before = Pt(4); p.paragraph_format.space_after = Pt(12)
    _font(p.add_run(f"Table {n}: "), 10.5, bold=True); _font(p.add_run(title), 10.5)


def _caption(doc, text, above=False):
    p = _para(doc, text, 10.5, False, False, align="center", after=3 if above else 10, before=6 if above else 0,
              keep=above)
    for r in p.runs:
        r.italic = False
    return p


def _figure(doc, path, caption, width=6.0):
    if os.path.exists(path):
        doc.add_picture(path, width=Inches(width))
        doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
        doc.paragraphs[-1].paragraph_format.keep_with_next = True
    _caption(doc, caption)


def _add_page_field(paragraph):
    run = paragraph.add_run(); _font(run, 11)
    for kind, txt in (("begin", None), (None, "PAGE"), ("end", None)):
        if kind:
            el = OxmlElement("w:fldChar"); el.set(qn("w:fldCharType"), kind)
        else:
            el = OxmlElement("w:instrText"); el.set(qn("xml:space"), "preserve"); el.text = txt
        run._r.append(el)


def _toc_line(doc, text, page, indent=0, bold=False):
    p = doc.add_paragraph()
    p.paragraph_format.left_indent = Cm(indent)
    p.paragraph_format.space_after = Pt(3)
    p.paragraph_format.tab_stops.add_tab_stop(Inches(6.2), WD_TAB_ALIGNMENT.RIGHT, WD_TAB_LEADER.DOTS)
    _font(p.add_run(f"{text}\t{page}"), 12, bold=bold)


def _list_table(doc, title, col_names, rows, widths):
    _para(doc, title, 15, True, align="center", after=12)
    _table(doc, col_names, rows, widths, size=11, left_cols=(1,))


PRICE_DEC = 2


def _f(v, d=None):
    d = PRICE_DEC if d is None else d
    if v is None or (isinstance(v, float) and v != v):
        return "N/A"
    return f"{v:,.{d}f}"


def _pct(v, d=2):
    return "N/A" if v is None or (isinstance(v, float) and v != v) else f"{v:,.{d}f}%"


# ------------------------------------------------------------------ the document
HEADINGS = [  # (label, level)
    ("1. Introduction", 0), ("1.1 Problem Statement", 1), ("1.2 Objectives", 1), ("1.3 Existing Similar Projects", 1),
    ("2. Proposed Methodology", 0), ("2.1 System Architecture", 1), ("2.2 Algorithm Description", 1),
    ("2.3 Implementation Flow", 1),
    ("3. Experimental Results", 0), ("3.1 Dataset Description", 1), ("3.2 Result Validation Parameters", 1),
    ("3.3 Result Analysis", 1), ("3.4 Result Discussion", 1), ("3.5 Implementation Screenshots", 1),
    ("3.6 Current Limitation", 1), ("3.7 Conclusion", 1), ("4. References", 0),
]
FIGS = [
    ("2.1", "System Architecture Diagram"), ("2.2", "Implementation Flow Chart of the Project"),
    ("3.1", "Closing Price with Train/Test Split and 30-Day Rolling Volatility"),
    ("3.2", "Return Distribution and Relationship with the Next-Day Return"),
    ("3.3", "Effect of Random Forest Hyperparameters on Cross-Validation Error"),
    ("3.4", "Actual versus Predicted Closing Price on the Test Period"),
    ("3.5", "Error Comparison: Naive, Default and Tuned Random Forest"),
    ("3.6", "Forecast Error Distribution and Errors over Time (Tuned Forest)"),
    ("3.7", "Random Forest Feature Importance"),
    ("3.8", "Next-Day Forecast with 90% Interval"),
]
TABS = [
    ("1.1", "Existing Price Forecasting Tools at a Glance"), ("3.1", "Dataset Summary"),
    ("3.2", "Training and Test Periods"), ("3.3", "Price and Volatility by Year"),
    ("3.4", "Input Features for the Random Forest"), ("3.5", "Five Sample Rows After Pre-Processing"),
    ("3.6", "Evaluation Metrics Used"), ("3.7", "Hyperparameter Tuning Results (5-Fold Time-Series Cross-Validation)"),
    ("3.8", "Default versus Tuned Random Forest on the Test Period"), ("3.9", "Up/Down Prediction Counts"),
    ("3.10", "Size of Forecast Errors: Tuned Forest and Naive Baseline"),
    ("3.11", "Random Forest Feature Importance"),
]


def _build(R, fig_dir, pages):
    global PRICE_DEC
    PRICE_DEC = R.get("price_dec", 2)
    pg = lambda key: str(pages.get(key, "-"))
    M = R["metrics"]; best = R["best_model"]; top = R["top_model"]; DEF = R["default_model"]
    naive, rt, rd = M["Naive (last value)"], M[top], M[DEF]
    asset, tick = R["asset_name"], R["ticker"]
    order = sorted(M, key=lambda k: M[k]["RMSE"])
    imp = (naive["RMSE"] - rt["RMSE"]) / naive["RMSE"] * 100          # tuned vs naive (RMSE)
    beat = rt["RMSE"] < naive["RMSE"]
    naive_best = best == "Naive (last value)"
    S = R["stats"]; n = R["test_n"]; wins = R["win_days"]
    dbest, margin = rt["DirAcc"], R["dir_margin"]
    dir_inside = abs(dbest - 50) <= margin
    cmax = max(R["corr"], key=lambda c: abs(c["corr"]))
    bp, dcv, bcv = R["best_params"], R["default_cv"], R["base_cv"]
    tt = R["tuning_top"]; best_cv = tt[0]
    gain_def_cv = (dcv["mse"] - best_cv["cv_mse"]) / dcv["mse"] * 100        # CV MSE: default -> tuned
    gain_base_cv = (bcv["mse"] - best_cv["cv_mse"]) / bcv["mse"] * 100       # CV MSE: 0% -> tuned
    mae_gain = (rd["MAE"] - rt["MAE"]) / rd["MAE"] * 100                      # test MAE: default -> tuned
    mse_gain = (rd["MSE"] - rt["MSE"]) / rd["MSE"] * 100
    synthetic = R["data_source_text"].startswith("SYNTHETIC")
    ND = R["next_day"]
    ci = sorted(R["coefs"], key=lambda c: -c["imp"])
    ND_pct = (ND["hi"] - ND["lo"]) / 2 / ND["pred"] * 100

    doc = Document()
    st = doc.styles["Normal"]; st.font.name = FONT; st.font.size = Pt(12)
    s0 = doc.sections[0]
    s0.page_width, s0.page_height = Cm(21), Cm(29.7)
    s0.top_margin = s0.bottom_margin = Cm(2.54); s0.left_margin = s0.right_margin = Cm(2.54)

    # ============ title page ============
    _para(doc, "", after=40)
    p = doc.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    _font(p.add_run("A Synopsis of "), 12); _font(p.add_run("Mini Project"), 12, bold=True)
    _para(doc, PROJECT_TITLE, 22, True, align="center", after=14, line=1.15)
    _para(doc, "Submitted to Manipal University, Jaipur", 11, align="center", after=2)
    _para(doc, "towards the partial fulfilment of the requirement for the award of the degree of", 10, align="center", after=10)
    _para(doc, "BACHELOR OF TECHNOLOGY", 12, True, align="center", after=8)
    _para(doc, "in", 13, align="center", after=8)
    _para(doc, "Computer and Communication Engineering", 13, align="center", after=8)
    _para(doc, SEMESTER, 12, align="center", after=14)
    _para(doc, "By:", 10, align="center", after=6)
    _para(doc, STUDENT_NAME, 12, True, align="center", after=2)
    _para(doc, REG_NO, 12, True, align="center", after=10)
    if LOGO_B64:
        open("_logo.png", "wb").write(base64.b64decode(LOGO_B64))
        doc.add_picture("_logo.png", width=Inches(3.0)); doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
    _para(doc, "Under the guidance of:", 11, align="center", after=6, before=14)
    _para(doc, GUIDE, 12, True, align="center", after=16)
    _para(doc, "Department of Computer and Communication Engineering", 14, True, align="center", after=6)
    _para(doc, "Manipal University Jaipur", 16, True, align="center")

    # ============ contents / lists ============
    p = _para(doc, "TABLE OF CONTENTS", 15, True, align="center", after=14); p.paragraph_format.page_break_before = True
    p = doc.add_paragraph(); p.paragraph_format.tab_stops.add_tab_stop(Inches(6.2), WD_TAB_ALIGNMENT.RIGHT)
    _font(p.add_run("TITLE\tPAGE NUMBER"), 12, bold=True)
    for label, lvl in HEADINGS:
        _toc_line(doc, label, pg(label), indent=0 if lvl == 0 else 0.8)
    p = _para(doc, "", after=0); p.paragraph_format.page_break_before = True
    _list_table(doc, "LIST OF FIGURES", ["FIGURE NUMBER", "FIGURE TITLE", "PAGE NUMBER"],
                [[n_, t, pg("Figure " + n_)] for n_, t in FIGS], [1.3, 4.2, 1.1])
    p = _para(doc, "", after=0); p.paragraph_format.page_break_before = True
    _list_table(doc, "LIST OF TABLES", ["TABLE NUMBER", "TABLE TITLE", "PAGE NUMBER"],
                [[n_, t, pg("Table " + n_)] for n_, t in TABS], [1.3, 4.2, 1.1])

    sec = doc.add_section(WD_SECTION.NEW_PAGE)
    sec.footer.is_linked_to_previous = False
    fp = sec.footer.paragraphs[0]; fp.alignment = WD_ALIGN_PARAGRAPH.CENTER; _add_page_field(fp)
    pgnum = OxmlElement("w:pgNumType"); pgnum.set(qn("w:start"), "1"); sec._sectPr.append(pgnum)

    # ---------------- Chapter 1 ----------------
    _para(doc, "CHAPTER 1: INTRODUCTION", 15, True, align="center", after=10, keep=True)
    _para(doc, (f"{asset} and the other cryptocurrencies trade on exchanges all day and all night, and the price can move by "
                "several percent within a few hours. Plenty of people follow these prices, from small traders to companies, "
                "so even a rough idea of where the price is going would be useful."))
    _para(doc, ("A machine learning model can learn from past prices. The catch is that prices come in time order, and the order "
                "matters. If the rows are shuffled before splitting them into training and test sets, the model gets to see the "
                "future while it trains, and the results look far better than they really are. So the data has to be split by time, "
                "and the model has to be tested on days that come after the days it learned from."))
    _para(doc, (f"In this project I take the daily closing price of {asset} ({tick}) and try to predict the price of the next day. "
                "I use Random Forest, which we studied in the AI and ML Lab. It is a group of decision trees whose predictions are averaged, "
                "so it can pick up non-linear patterns without much preparation of the data, and it can also tell us which features it used most. "
                "I tune its hyperparameters with cross-validation and compare it with a very simple baseline, so I can see whether the forest really adds anything."))

    _h2(doc, "1.1 Problem Statement")
    _para(doc, "Forecasting a cryptocurrency price is difficult, and these are the main reasons:", align="left")
    _numbered(doc, [
        ("Wild price swings", f"The daily return of {asset} in my data has a standard deviation of {S['ret_std']*100:.2f}%, and big "
         "jumps are normal. News, new rules and the mood of the market move the price, and none of that shows up in "
         "past prices."),
        ("Prices that keep drifting", "The price goes up and down over long stretches, so a model trained on one period can face "
         "completely different values later. Predicting the daily percentage change instead of the price makes the data more stable."),
        ("Peeking at the future", "If the rows are shuffled, or a feature uses tomorrow's information by mistake, the results look "
         "much better than they really are. The split therefore has to follow time, and each prediction must be made one "
         "day ahead using only the past."),
        ("Over-fitting and a hard baseline", "A forest of deep trees can memorise the noise in the training data, so its settings have to be tuned. "
         "And because prices behave a lot like a random walk, the simple forecast “tomorrow will be the same as today” is already pretty good. "
         "The forest is only worth using if it beats it."),
    ])

    _h2(doc, "1.2 Objectives")
    _para(doc, "Based on the problems above, I set these objectives for the project:", align="left")
    _bullets(doc, [
        f"Load the daily price history of {asset} from a Kaggle CSV file, check and clean it, and study it with plots",
        "Build a Random Forest regressor that predicts the next-day return, and turn that into a price forecast",
        "Make features that only use information available before the day being predicted",
        "Tune n_estimators, max_depth and min_samples_leaf with cross-validation that respects the time order, and measure the improvement in MAE and MSE",
        "Test the tuned forest day by day, and compare it with the default forest and with a naive baseline using MAE, MSE, RMSE, MAPE and a confusion matrix of up/down calls",
        "Use the feature importance to see what the forest relies on, and give a next-day forecast with an interval",
    ])

    _h2(doc, "1.3 Existing Similar Projects")
    _para(doc, "Many tools and notebooks already try to forecast crypto prices. Table 1.1 lists a few of them, going by what "
               "their own pages say about them.", after=4)
    _table(doc, ["Project / Tool", "Technology Used", "Features", "Limitations"], [
        ["TradingView indicators", "Technical analysis (moving averages, RSI, MACD)", "Charts and trading signals for every coin",
         "Only describes past prices, depends on the trader's judgement, no measured forecast error"],
        ["Prophet (Meta)", "Additive trend and seasonality model", "Quick forecasts that handle holidays and seasons",
         "Expects smooth trends, which crypto rarely has"],
        ["Deep-learning notebooks (Kaggle)", "LSTM / GRU in TensorFlow", "Can pick up non-linear patterns",
         "Many use random splits that leak information, are hard to explain and skip the naive baseline"],
        ["Price-prediction websites", "Rules or opinions", "Quick yearly price targets",
         "No testing, no stated error, cannot be reproduced"],
    ], [1.35, 1.55, 1.6, 2.1], size=10, left_cols=(0, 1, 2, 3))
    _tcap(doc, "1.1")

    # ---------------- Chapter 2 ----------------
    _chapter(doc, "CHAPTER 2: PROPOSED METHODOLOGY")
    _h2(doc, "2.1 System Architecture")
    _para(doc, "I split the work into five stages so that any one of them can be changed without touching the others.")
    _figure(doc, f"{fig_dir}/arch.png", "Figure 2.1: System Architecture Diagram", 6.4)
    _bullets(doc, [
        "Data pre-processing loads the daily prices from the Kaggle CSV file, puts them on a continuous daily calendar and works out the daily returns.",
        "Feature engineering builds nine features from the past returns: lagged returns, rolling averages and rolling volatility.",
        "Random Forest training and tuning fits a default forest and then searches for the best n_estimators, max_depth and min_samples_leaf with time-series cross-validation.",
        "Evaluation predicts each test day one step ahead, adds the true value to the history and moves on to the next day.",
        "Forecasting measures the errors, shows which features the forest used and gives a next-day forecast.",
    ])

    _h2(doc, "2.2 Algorithm Description")
    _para(doc, "Let P(t) be the closing price on day t. The forest does not predict the price directly. It predicts the next-day return "
               "r(t+1) = ln P(t+1) − ln P(t), and the price forecast is P(t) · exp(r̂(t+1)). The algorithm works like this:")
    _rich(doc, [("Decision tree: ", True), ("a tree asks a series of yes/no questions about the features (for example, was yesterday's return above 2%?). "
               "Each answer sends the day down one branch until it reaches a leaf, and the leaf's prediction is the average target of the training days that ended up there. "
               "A very deep tree fits the training data almost perfectly but also memorises its noise.", False)])
    _rich(doc, [("Bagging: ", True), ("each tree of the forest is trained on a random sample of the training days (drawn with replacement), so every tree sees slightly different data.", False)])
    _rich(doc, [("Random feature choice: ", True), ("at every split a tree only looks at a random subset of the features, which makes the trees different from each other.", False)])
    _rich(doc, [("Averaging: ", True), ("the forest's prediction is the average of the B trees, r̂(x) = (1/B) Σ Tᵦ(x). Averaging many different trees cancels out much of their noise, so the forest over-fits less than a single tree.", False)])
    _rich(doc, [("Hyperparameters: ", True), ("n_estimators is the number of trees B, max_depth limits how deep a tree can grow and min_samples_leaf is the smallest number of days allowed in a leaf. "
               "Small depth and large leaves make a simpler forest; large depth and small leaves make a more flexible one.", False)], after=6)
    _para(doc, "I carried out the project in the following steps.", align="left", keep=True)
    steps = [
        ("1. Data Loading and Validation", f"The data has {R['n_obs']:,} daily records of {tick} from {R['start']} to {R['end']}. "
         "I checked it for missing values, duplicate dates and gaps. Any missing day was filled with the previous day's price so that every calendar day is present."),
        ("2. Daily Returns", "The price drifts over the years, so I work with the daily return instead (the percentage change from one day to the next). "
         "Its mean and spread stay much more stable over time."),
        ("3. Features Without Leakage", "Every row only uses what was known at the end of day t: five lagged returns, the 7-day and 14-day average return "
         "and the 7-day and 14-day volatility. The target is the return of day t+1. A forest does not need the features to be scaled."),
        ("4. Split by Time", f"The first {R['train_n']:,} days ({R['start']} to {R['train_end']}) are for training and the last "
         f"{R['test_n']} days ({R['test_start']} to {R['end']}) are for testing. The data is never shuffled."),
        ("5. Hyperparameter Tuning", "I tried 2 values of n_estimators, 5 of max_depth and 3 of min_samples_leaf (30 combinations) and scored each one with 5-fold time-series cross-validation. "
         "In every fold the forest trains on an earlier part of the training period and is scored on the part that follows, so it never trains on the future. "
         "The scores are the MSE and MAE of the next-day return, and the combination with the lowest MSE is kept."),
        ("6. Day-by-Day Testing", f"For each of the {R['test_n']} test days the forest predicts one day ahead, and then the real price is added to the "
         "history. To save time the forest is refitted every 5 days. The default forest (100 trees, no depth limit, min_samples_leaf = 1) and the tuned forest are tested in exactly the same way."),
        ("7. Evaluation and Interpretation", "I calculate MAE, MSE, RMSE, MAPE and the number of correct up/down calls, count the days on which the forest beat the naive forecast, "
         "and read the feature importance of the forest."),
    ]
    for h, b in steps:
        _para(doc, h, 12, True, align="left", after=1, keep=True)
        _para(doc, b)

    _h2(doc, "2.3 Implementation Flow")
    _figure(doc, f"{fig_dir}/flow.png", "Figure 2.2: Implementation Flow Chart of the Project", 5.2)

    # ---------------- Chapter 3 ----------------
    _chapter(doc, "CHAPTER 3: EXPERIMENTAL RESULTS")
    _h2(doc, "3.1 Dataset Description")
    if synthetic:
        _para(doc, "NOTE: this draft was generated from SYNTHETIC stand-in data. Re-run the notebook with the real CSV file to get the real report.", 11, True)
    _para(doc, (f"I used the daily price data of {tick}, {R['data_source_text']} "
                f"There are {R['n_obs']:,} daily records from {R['start']} to {R['end']}. Only the closing price is forecast. "
                "Table 3.1 gives the main numbers."), after=4)
    _table(doc, ["Attribute", "Description", "Values/Range"], [
        ["Asset", "Cryptocurrency and ticker", f"{asset} ({tick})"],
        ["Total Records", "Number of daily observations", f"{R['n_obs']:,} rows"],
        ["Time Period", "Data collection span", f"{R['start']} – {R['end']}"],
        ["Columns", "Fields per day", "Date, Price, Open, High, Low, Vol, Change %"],
        ["Minimum / Maximum close", "Price range (USD)", f"{_f(S['min'])} / {_f(S['max'])}"],
        ["Mean / Std. deviation", "Closing price (USD)", f"{_f(S['mean'])} / {_f(S['std'])}"],
        ["Daily return", "Mean and volatility", f"{S['ret_mean']*100:.3f}% / {S['ret_std']*100:.3f}%"],
        ["Biggest daily gain / fall", "Extreme days", f"{S['max_gain']:+.1f}% / {S['max_fall']:+.1f}%"],
        ["Data Quality", "Missing days / duplicates", f"{R['filled_days']} filled / {R.get('dup_days', 0)} duplicates"],
    ], [1.9, 2.4, 2.2])
    _tcap(doc, "3.1")

    sp = R["split_table"]
    _table(doc, ["Data Set", "Period", "Days", "Mean Close (USD)", "Mean Daily Return", "Volatility"],
           [[r["name"], r["period"], f"{r['days']:,}", _f(r["mean_close"]), f"{r['mean_ret']*100:.3f}%", f"{r['vol']*100:.3f}%"] for r in sp],
           [0.9, 1.9, 0.7, 1.2, 1.0, 0.8], size=10)
    _tcap(doc, "3.2")

    _table(doc, ["Year", "Days", "First Close (USD)", "Last Close (USD)", "Change", "Daily Volatility"],
           [[r["year"], r["days"], _f(r["first"]), _f(r["last"]), _pct(r["chg"], 1), f"{r['vol']*100:.2f}%"] for r in R["year_table"]],
           [0.7, 0.7, 1.4, 1.4, 1.0, 1.3], size=10)
    _tcap(doc, "3.3")

    _para(doc, "Before training I did the following:", after=2)
    _bullets(doc, [
        "Parsed the dates, sorted the rows from oldest to newest and put the series on a continuous daily calendar.",
        "Filled the missing days with the previous day's price.",
        "Worked out the daily returns and built every feature from data up to day t only, without shuffling the rows.",
        "Kept the features unscaled, because a Random Forest splits on thresholds and is not affected by the scale of a feature.",
    ])
    _para(doc, "The forest uses 9 features, listed in Table 3.4.", after=4)
    _table(doc, ["Group", "Features", "Description"], [
        ["Lagged returns", "lag1, lag2, lag3, lag4, lag5", "Log-return of today and of the four days before"],
        ["Rolling mean", "rm7, rm14", "Average daily return over the last 7 and 14 days (short-term momentum)"],
        ["Rolling volatility", "rs7, rs14", "Standard deviation of returns over the last 7 and 14 days (how nervous the market is)"],
    ], [1.4, 2.2, 2.9], left_cols=(0, 1, 2))
    _tcap(doc, "3.4")
    _para(doc, "Table 3.5 shows five days from the start of the test period, exactly as the Colab notebook prints them after "
               "pre-processing. All the features are known at the end of the day. The last column, next_ret, is what the "
               "forest has to predict.", after=4)
    _table(doc, ["Date", "Close", "lag1", "lag2", "rm7", "rs7", "next_ret"],
           [[r["date"], _f(r["close"]), f"{r['lag1']:.4f}", f"{r['lag2']:.4f}", f"{r['rm7']:.4f}", f"{r['rs7']:.4f}", f"{r['next_ret']:.4f}"]
            for r in R["sample_rows"]], [1.1, 1.1, 0.9, 0.9, 0.9, 0.9, 0.9], size=10, left_cols=())
    _tcap(doc, "3.5")

    _h2(doc, "3.2 Result Validation Parameters")
    _para(doc, "The price is a number and not a class, so I use error measures here, as in the Random Forest experiments of the lab. I also "
               "count how often the forest gets the direction of the move right (up or down) and show it as a confusion matrix, since a forecast "
               "that knows the direction is useful even when the exact value is a bit off.", after=4)
    _table(doc, ["Metric", "Formula", "Purpose"], [
        ["MAE", "(1/N) Σ |y − ŷ|", "Average error in USD"],
        ["MSE", "(1/N) Σ (y − ŷ)²", "Average squared error, large misses count more"],
        ["RMSE", "√ MSE", "Error in USD, easier to read than MSE"],
        ["MAPE", "(100/N) Σ |y − ŷ| / y", "Average error as a percentage of the price"],
        ["Directional Accuracy", "Share of days with the right up/down call", "Is the direction right? 50% is the same as guessing"],
    ], [1.5, 2.8, 2.2], left_cols=(0, 1, 2))
    _tcap(doc, "3.6")

    _h2(doc, "3.3 Result Analysis")
    _para(doc, "The hyperparameters were chosen on the training period only, with 5-fold time-series cross-validation. Table 3.7 shows the eight "
               "best of the 30 combinations, with the cross-validation MSE and MAE of the next-day return (in %² and %). The last two rows are references: "
               "the default forest, and a model that always predicts a 0% return.", after=4)
    rows = [[t["n_estimators"], t["max_depth"], t["min_samples_leaf"], f"{t['cv_mse']:.3f}", f"{t['cv_mae']:.3f}"] for t in tt]
    rows += [["100", "None", "1", f"{dcv['mse']:.3f}", f"{dcv['mae']:.3f}"], ["-", "-", "-", f"{bcv['mse']:.3f}", f"{bcv['mae']:.3f}"]]
    rows[len(tt)][0] = "100 (default)"; rows[len(tt) + 1][0] = "Always predict 0%"
    _table(doc, ["n_estimators", "max_depth", "min_samples_leaf", "CV MSE (%²)", "CV MAE (%)"], rows, [1.5, 1.0, 1.4, 1.2, 1.1], bold_rows=(0,), left_cols=())
    _tcap(doc, "3.7")
    cvtxt = (f"The best combination is n_estimators = {bp['n_estimators']}, max_depth = {bp['max_depth']} and min_samples_leaf = {bp['min_samples_leaf']}, with a CV MSE of {best_cv['cv_mse']:.3f}. "
             f"That is {abs(gain_def_cv):.1f}% {'lower' if gain_def_cv > 0 else 'higher'} than the default forest ({dcv['mse']:.3f}) "
             + (f"and {abs(gain_base_cv):.1f}% {'lower' if gain_base_cv > 0 else 'higher'} than always predicting 0% ({bcv['mse']:.3f}). " ))
    _para(doc, cvtxt + f"After tuning, I tested the default and the tuned forest on the last {n} days, one day ahead at a time (Table 3.8).")
    _table(doc, ["Model", "MAE (USD)", "MSE (USD²)", "RMSE (USD)", "MAPE", "Directional Acc."],
           [[k, _f(M[k]["MAE"]), f"{M[k]['MSE']:.6f}", _f(M[k]["RMSE"]), _pct(M[k]["MAPE"]), _pct(M[k]["DirAcc"], 1)] for k in order],
           [1.9, 0.9, 1.0, 0.9, 0.7, 1.1], bold_rows=(0,), size=10)
    _tcap(doc, "3.8")
    _para(doc, (f"Compared with the default forest, tuning changed the test MAE by {abs(mae_gain):.1f}% ({'lower' if mae_gain > 0 else 'higher'}) and the test MSE by {abs(mse_gain):.1f}% "
                f"({'lower' if mse_gain > 0 else 'higher'}). " +
                (f"The naive last-value forecast has the lowest RMSE ({_f(naive['RMSE'])} USD); the tuned forest is {abs(imp):.2f}% behind it. " if naive_best else
                 f"{best} has the lowest RMSE ({_f(M[best]['RMSE'])} USD). The naive last-value forecast has an RMSE of {_f(naive['RMSE'])} USD, so the tuned forest is {abs(imp):.2f}% "
                 f"{'lower' if imp > 0 else 'higher'}. ") +
                f"The tuned forest was closer to the real price than the naive forecast on {wins} of the {n} test days."))
    d = R["dir_conf"]
    _para(doc, f"Table 3.9 shows how often the tuned forest called the direction of the daily move correctly.", after=4)
    tu, td = d["uu"] + d["ud"], d["du"] + d["dd"]
    _table(doc, ["Actual \\ Predicted", "Down", "Up", "Total", "Recall (%)"], [
        ["Down", d["dd"], d["du"], td, _f(d["dd"] / td * 100 if td else float("nan"), 1)],
        ["Up", d["ud"], d["uu"], tu, _f(d["uu"] / tu * 100 if tu else float("nan"), 1)],
        ["Precision (%)", _f(d["dd"] / (d["dd"] + d["ud"]) * 100 if (d["dd"] + d["ud"]) else float("nan"), 1),
         _f(d["uu"] / (d["uu"] + d["du"]) * 100 if (d["uu"] + d["du"]) else float("nan"), 1), tu + td,
         f"{dbest:.1f} (Accuracy)"],
    ], [1.5, 1.0, 1.0, 1.0, 1.6])
    _tcap(doc, "3.9")
    E = R["err_table"]
    _para(doc, "Table 3.10 puts the size of the errors of the tuned forest next to the naive baseline.", after=4)
    _table(doc, ["Measure", "Tuned forest", "Naive (last value)"], [
        ["Mean error (bias), USD", _f(E[top]["bias"]), _f(E["Naive (last value)"]["bias"])],
        ["Standard deviation of error, USD", _f(E[top]["std"]), _f(E["Naive (last value)"]["std"])],
        ["Largest absolute error, USD", _f(E[top]["max"]), _f(E["Naive (last value)"]["max"])],
        ["Days within 1% of actual price", _pct(E[top]["w1"], 1), _pct(E["Naive (last value)"]["w1"], 1)],
        ["Days within 2% of actual price", _pct(E[top]["w2"], 1), _pct(E["Naive (last value)"]["w2"], 1)],
        ["Days within 5% of actual price", _pct(E[top]["w5"], 1), _pct(E["Naive (last value)"]["w5"], 1)],
    ], [3.0, 1.8, 1.7])
    _tcap(doc, "3.10")
    C = R["coefs"]
    _para(doc, "Table 3.11 lists the feature importance of the tuned forest (each feature's share of the total, in %), next to the plain correlation "
               "of that feature with the next-day return.", after=4)
    _table(doc, ["Feature", "RF Importance (%)", "Correlation with Next-Day Return"],
           [[c["feature"], f"{c['imp']:.1f}", f"{c['corr']:+.3f}"] for c in C], [1.6, 2.0, 2.6])
    _tcap(doc, "3.11")

    _h2(doc, "3.4 Result Discussion")
    _para(doc, "Here is what the results show.", after=4)
    _para(doc, "1. How accurate the forecasts are", 12, True, align="left", after=1, keep=True)
    _para(doc, (f"Over the last {n} days the tuned forest had a MAPE of {_pct(rt['MAPE'])} (RMSE {_f(rt['RMSE'])} USD), so a typical one-day miss is about "
                f"{rt['MAPE']:.1f}% of the price. " +
                (f"Its RMSE is {abs(imp):.1f}% lower than the naive baseline, and it was closer on {wins} of {n} days. " if beat else
                 f"That is not better than the naive baseline: its RMSE is {abs(imp):.1f}% higher, and it was closer than the naive forecast on only {wins} of {n} days. ") +
                f"Table 3.10 shows that the two forecasts have errors of almost the same size."))
    _para(doc, "2. What tuning achieved", 12, True, align="left", after=1, keep=True)
    _para(doc, (f"In cross-validation the tuned forest was {abs(gain_def_cv):.1f}% {'better' if gain_def_cv > 0 else 'worse'} than the default forest in MSE, so the default settings were clearly "
                + ("too flexible: a forest of fully grown trees fits noise." if gain_def_cv > 1 else "not much different from the tuned ones.") + " "
                f"The best settings are at the simple end of the grid (max_depth = {bp['max_depth']}, min_samples_leaf = {bp['min_samples_leaf']}), "
                f"and Figure 3.3 shows that the error rises as the trees grow deeper. On the test days, tuning changed the MAE by {abs(mae_gain):.1f}% and the MSE by {abs(mse_gain):.1f}% "
                f"({'an improvement' if mse_gain > 0 else 'a small loss'}). The number of trees matters very little."))
    d = R["dir_conf"]
    _para(doc, "3. Getting the direction right", 12, True, align="left", after=1, keep=True)
    _para(doc, (f"The tuned forest called the direction right on {dbest:.1f}% of the days. If someone guessed up or down by flipping a coin, over {n} days they would "
                f"usually land between about {50-margin:.0f}% and {50+margin:.0f}%. " +
                ("That is inside this range, so I cannot say the forest is better than luck. " if dir_inside else
                 "That is outside this range, which is a good sign, but the test is short, so I would not rely on it. ") +
                (f"Table 3.9 also shows that the forest called \u201cdown\u201d on {d['dd'] + d['ud']} of the {n} days, so its score mostly reflects how many days were really down days. " if (d['dd'] + d['ud']) / n > 0.7 or (d['dd'] + d['ud']) / n < 0.3 else "")
                + "A small error in dollars also does not mean the model knows whether the price will go up or down."))
    _para(doc, "4. Why the forest struggles", 12, True, align="left", after=1, keep=True)
    _para(doc, (f"The correlation between each feature and the next-day return is very small. The largest is {cmax['corr']:+.3f} for {cmax['feature']} "
                "(Figure 3.2), so yesterday's returns and volatility say very little about tomorrow's return. "
                f"This matches Table 3.7: the best tuned forest has a CV MSE only {abs(gain_base_cv):.1f}% "
                + ("below" if gain_base_cv > 0 else "above") + " that of a model that always predicts 0%. When the features carry so little signal, even a well tuned forest cannot do much better than the baseline."))
    _para(doc, "5. What the forest relies on", 12, True, align="left", after=1, keep=True)
    top_corr = next(c["corr"] for c in R["corr"] if c["feature"] == ci[0]["feature"])
    spread_txt = (f"One feature stands out: {ci[0]['feature']} carries about {ci[0]['imp']:.0f}% of the total, and the rest is spread over many features. "
                  if ci[0]["imp"] >= 30 else "The importance is spread over many features and no single one dominates. ")
    _para(doc, (f"The most important feature is {ci[0]['feature']} ({ci[0]['imp']:.1f}% of the total), followed by {ci[1]['feature']} ({ci[1]['imp']:.1f}%) and {ci[2]['feature']} ({ci[2]['imp']:.1f}%) (Figure 3.7). "
                + spread_txt + f"Even so, the correlation of {ci[0]['feature']} with the next-day return is only {top_corr:+.3f}, so it is not a strong predictor. "
                "Importance only shows which features the forest used to split the days; it does not say whether a feature moves the price up or down."))
    _para(doc, "6. The next-day forecast", 12, True, align="left", after=1, keep=True)
    _para(doc, (f"After the last date in the file ({R['end']}, close {_f(ND['last'])} USD), the tuned forest forecasts {_f(ND['pred'])} USD for {ND['date']}. "
                f"Using the 5th and 95th percentile of its errors on the test days, the 90% interval is {_f(ND['lo'])} to {_f(ND['hi'])} USD, about ±{ND_pct:.1f}% (Figure 3.8). "
                "The interval is more useful than the single number, because it shows how unsure the forecast is."))
    _para(doc, ("In short, tuning makes the forest more sensible, but the forest and the naive baseline end up almost level. "
                "It is a useful benchmark, not a tool for trading."))

    _h2(doc, "3.5 Implementation Screenshots")
    _para(doc, "These figures come straight from the Python code in Google Colab.")
    for n_, t in FIGS[2:]:
        _figure(doc, f"{fig_dir}/fig_{n_.replace('.', '_')}.png", f"Figure {n_}: {t}", 6.0)

    _h2(doc, "3.6 Current Limitation")
    _para(doc, "The forest works as a fair benchmark, but it has some weak points.")
    lim = [
        ("1. Things the data cannot see.", ("Prices react to news, new rules, exchange failures and social media, and none of that is in the series. " +
          ("XRP in particular has jumped and fallen sharply on news about the legal case between Ripple and the US regulator. " if asset.upper().startswith("XRP") else "") +
          f"Even the tuned forest misses by about {rt['MAPE']:.1f}% a day and its direction calls are {'close to chance' if dir_inside else 'only somewhat better than chance'}.")),
        ("2. Changing market conditions.", "Volatility and trend are very different in a bull market and a bear market, so a pattern from an older year may not hold today. This is why the forest is refitted as the test goes on."),
        ("3. Only the price is used.", "I did not use trading volume, sentiment or economic variables. Adding them might help, and without them there is a limit to what the forest can learn."),
        ("4. Model limits.", "A Random Forest cannot predict values outside the range it saw in training, so it cannot follow a sudden crash or rally by itself. "
          "The 90% interval is based on only 90 test days, so it is rough, and the importance scores do not show the direction of an effect."),
    ]
    for h, b in lim:
        _rich(doc, [(h + " ", True), (b, False)], after=5)

    _h2(doc, "3.7 Conclusion")
    _para(doc, (f"In this project I built a Random Forest that predicts the next-day price of {asset} using only information that was available before each "
                f"day. Tuning n_estimators, max_depth and min_samples_leaf with time-series cross-validation gave a CV MSE {abs(gain_def_cv):.1f}% {'lower' if gain_def_cv > 0 else 'higher'} than the default forest. "
                f"On the last {n} days the tuned forest had a MAPE of {_pct(rt['MAPE'])}, " +
                (f"{abs(imp):.1f}% below the naive baseline in RMSE, " if beat else f"{abs(imp):.1f}% behind the naive baseline in RMSE, ") +
                (f"and the direction was right on {dbest:.0f}% of days, which is within the range of luck. " if dir_inside else f"and the direction was right on {dbest:.0f}% of days, a little above luck, but the test is short. ") +
                "The tiny correlations and the cross-validation results both say the same thing: daily crypto returns are very hard to predict from past prices alone. "
                "If I had more time I would add features such as trading volume and sentiment, try other coins, and test longer periods. "
                "These results are for a lab exercise and are not financial advice."))

    # ---------------- Chapter 4 ----------------
    _chapter(doc, "CHAPTER 4: REFERENCES")
    refs = [
        "Breiman, L. (2001). Random forests. Machine Learning, 45(1), 5–32. https://doi.org/10.1023/A:1010933404324",
        "Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. Journal of Machine Learning Research, 12, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html",
        "Géron, A. (2017). Hands-On Machine Learning with Scikit-Learn and TensorFlow. O’Reilly Media.",
        "Raschka, S., & Mirjalili, V. (2017). Python Machine Learning (2nd ed.). Packt Publishing.",
        "Hastie, T., Tibshirani, R., & Friedman, J. (2009). The Elements of Statistical Learning (2nd ed.). Springer.",
        "Kaufman, S., Rosset, S., Perlich, C., & Stitelman, O. (2012). Leakage in data mining: Formulation, detection, and avoidance. ACM Transactions on Knowledge Discovery from Data, 6(4), 1–21. https://doi.org/10.1145/2382577.2382579",
        "Fama, E. F. (1970). Efficient capital markets: A review of theory and empirical work. The Journal of Finance, 25(2), 383–417.",
        ("Schwartz, D., Youngs, N., & Britto, A. (2014). The Ripple protocol consensus algorithm. Ripple Labs white paper." if asset.upper().startswith("XRP")
         else "Nakamoto, S. (2008). Bitcoin: A peer-to-peer electronic cash system. https://bitcoin.org/bitcoin.pdf"),
        "McKinney, W. (2010). Data structures for statistical computing in Python. Proceedings of the 9th Python in Science Conference, 56–61.",
        "Harris, C. R., et al. (2020). Array programming with NumPy. Nature, 585, 357–362. https://doi.org/10.1038/s41586-020-2649-2",
        "Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. Computing in Science & Engineering, 9(3), 90–95. https://doi.org/10.1109/MCSE.2007.55",
        (f"{R['dataset_title']}." + (f" {R['dataset_url']}" if R.get('dataset_url') else "")),
    ]
    for r in refs:
        p = doc.add_paragraph(style="List Bullet"); _font(p.add_run(r), 11.5)
        p.paragraph_format.space_after = Pt(4); p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    return doc


def _find_pages(pdf_path):
    from pypdf import PdfReader
    pages = [re.sub(r"\s+", " ", (p.extract_text() or "")) for p in PdfReader(pdf_path).pages]
    start = next(i for i, t in enumerate(pages) if "CHAPTER 1: INTRODUCTION" in t)
    found = {}
    keys = [(h, h) for h, _ in HEADINGS] + [("Figure " + n, "Figure " + n + ":") for n, _ in FIGS] + [("Table " + n, "Table " + n + ":") for n, _ in TABS]
    for key, needle in keys:
        for i in range(start, len(pages)):
            t = pages[i]
            if key[0] in "123456789" and not key.startswith(("Fig", "Tab")):
                # headings: chapters are written as "CHAPTER n: TITLE"
                if re.match(r"^\d\. ", key):
                    num, title = key.split(". ", 1)
                    ok = f"CHAPTER {num}: {title.upper()}" in t
                else:
                    ok = needle in t
            else:
                ok = needle in t
            if ok:
                found[key] = i - start + 1
                break
    return found


def _to_pdf(docx_path):
    exe = shutil.which("soffice") or shutil.which("libreoffice")
    if not exe:
        return None
    subprocess.run([exe, "--headless", "--convert-to", "pdf", "--outdir", os.path.dirname(os.path.abspath(docx_path)) or ".", docx_path],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=300)
    pdf = os.path.splitext(docx_path)[0] + ".pdf"
    return pdf if os.path.exists(pdf) else None


def build_report(results_path="results.json", fig_dir="figures", out_path="AIML_Report_Crypto_Price_Prediction.docx"):
    R = json.load(open(results_path))
    doc = _build(R, fig_dir, {})
    doc.save(out_path)
    pdf = _to_pdf(out_path)
    if pdf:
        try:
            pages = _find_pages(pdf)
            doc = _build(R, fig_dir, pages)          # second pass with real page numbers
            doc.save(out_path)
            pdf = _to_pdf(out_path)
        except Exception as e:
            print("Page-number pass skipped:", e)
    else:
        print("LibreOffice not found: page numbers in the contents list are left as '-'.")
    return out_path, pdf


REG_NO = "______________"      # <- type your registration number here
GUIDE  = "Dr Sandeep Kumar Sharma"   # <- change if your guide is different
LOGO_B64 = "..."
docx_path, pdf_path = build_report("results.json", "figures", "AIML_Report_XRP_RF.docx")
print("Report created:", docx_path, pdf_path)
try:
    from google.colab import files
    files.download(docx_path)
    if pdf_path: files.download(pdf_path)
except ImportError:
    pass

---
### Notes
* Forecasts of crypto prices are uncertain; this notebook is an academic exercise, not financial advice.
* Every forest is tested with **walk-forward validation**: a day is never predicted with information from that day or later.